---

## Problema 3: Estimar la difusividad térmica: problema inverso


Se considera la ecuación de difusión unidimensional:
$$
u_t = \alpha u_{xx}
$$

donde $u(x,t)$ representa la temperatura y $\alpha > 0$ es la difusividad térmica desconocida. Supongamos que queremos estimar la solución con una PINN, en el dominio $x \in [0,1]$ y $t \in[0,1]$, con condiciones iniciales $  u(x,0) = \sin(\pi x)$, $u(0,t) = 0$ y $u(1,t) = 0$.

Cargar los datos observados de `data/difusion_1D.csv`. Contiene valores de $(x, t, u(x,t)$ para varios pares $(x,t)$ en el dominio mencionado.  

1. Utilizando `dde.icbc.DirichletBC` para las condiciones de contorno, `dde.icbc.IC` para las condiciones iniciales, y `dde.icbc.PointSetBC` para los puntos de colocacion provistos,  entrenar una PINN dependiente del tiempo con `dde.data.TimePDE` que estime el valor del parámetro $\alpha$. TIP: estas clases están pensadas para trabajar con numpy arrays.

2. Entrenar el modelo durante 5000 épocas con el optimizador Adam, y luego utilizar L-BFGS para el refinamiento de la solución.  

In [ ]:
# Importar bibliotecas necesarias
import pandas as pd  # Para manipulación de datos en DataFrames
import deepxde as dde  # Librería para PINNs (Physics-Informed Neural Networks)
import numpy as np  # Para operaciones numéricas

# Cargar los datos observados del archivo CSV
# El archivo contiene valores de (x, t, u(x,t)) para puntos en el dominio espaciotemporal
data = pd.read_csv("data/difusion_1D.csv")

In [ ]:
# ============================================================================
# DEFINICIÓN DE CONDICIONES DE FRONTERA E INICIALES
# ============================================================================

# Función que identifica si un punto está en la frontera del dominio
def boundary(x, on_boundary):
    return on_boundary

# Función que define el valor de la condición de Dirichlet en la frontera
def boundary_value(x):
    return 0.0

# Función que define la condición inicial
def u0(x):
    return np.sin(np.pi * x[:, 0:1])

# ============================================================================
# DEFINICIÓN DE LA ECUACIÓN DIFERENCIAL PARCIAL (PDE)
# ============================================================================

def pde(x, u, alpha):
    # Calcular la derivada temporal de u respecto a t (componente j=1)
    u_t = dde.grad.jacobian(u, x, i=0, j=1)
    
    # Calcular la segunda derivada espacial de u respecto a x (componente i=0)
    u_xx = dde.grad.hessian(u, x, i=0, j=0)
    
    # Retornar el residuo de la PDE
    return u_t - alpha * u_xx

# ============================================================================
# DEFINICIÓN DEL DOMINIO ESPACIOTEMPORAL
# ============================================================================

# Crear el dominio espacial: x ∈ [0, 1]
geom = dde.geometry.Interval(0.0, 1.0)

# Crear el dominio temporal: t ∈ [0, 1]
time_domain = dde.geometry.TimeDomain(0.0, 1.0)

# Combinar los dominios espacial y temporal
# Esto crea un dominio bidimensional (x, t)
geomtime = dde.geometry.GeometryXTime(geom, time_domain)

# ============================================================================
# DEFINICIÓN DE CONDICIONES DE CONTORNO E INICIALES (BC/IC)
# ============================================================================

# Condición de contorno de Dirichlet: u = 0 en x = 0 y x = 1 para todo t
bc = dde.icbc.DirichletBC(geomtime, boundary_value, boundary)

# Condición inicial: u(x, 0) = sin(π*x)
ci = dde.icbc.IC(geomtime, u0, lambda _, on_initial: on_initial)

# Puntos de colocación basados en los datos observados cargados del CSV
# Estos puntos contienen las observaciones experimentales
pointset_bc = dde.icbc.PointSetBC(
    data[["x", "t"]].values,  # Coordenadas (x, t) de los datos
    data["u"].values.reshape(-1, 1),  # Valores observados de u(x, t)
    component=0,  # Componente de la solución (para sistemas escalares)
)

# ============================================================================
# DEFINICIÓN DEL PARÁMETRO ENTRENABLE
# ============================================================================

# Definir α como una variable entrenable de DeepXDE
# Se inicializa con un valor de 0.1 y será ajustado durante el entrenamiento
alpha = dde.Variable(0.1)

# ============================================================================
# CONSTRUCCIÓN DEL PROBLEMA DE DATOS (DATA)
# ============================================================================

# Crear el problema PDE con dependencia temporal
# TimePDE organiza automáticamente los puntos de colocación en el dominio espaciotemporal
data = dde.data.TimePDE(
    geomtime,  # Dominio espaciotemporal
    lambda x, u: pde(x, u, alpha),  # Función que define la PDE
    [bc, ci, pointset_bc],  # Condiciones de contorno, iniciales y puntos de datos
    num_domain=100,  # Número de puntos de colocación en el dominio interior
    num_boundary=40,  # Número de puntos de colocación en la frontera
    num_initial=20,  # Número de puntos de colocación en la condición initial
)

In [ ]:
# ============================================================================
# VISUALIZACIÓN DE LOS PUNTOS DE MUESTREO EN EL DOMINIO ESPACIOTEMPORAL
# ============================================================================

import matplotlib.pyplot as plt

# Crear figura y eje para la visualización
fig, ax = plt.subplots(figsize=(6, 6))

# Dibujar la geometría del dominio espacial (intervalo [0, 1] en x)
ax.plot([0, 1], [0, 0], "k-", lw=2, label="Geometría")

# Obtener un lote de datos de entrenamiento para visualización
data.train_next_batch()

# Extraer todas las coordenadas de puntos de colocación 
points = data.train_x

# Dibujar los puntos de colocación en el plano (x, t)
ax.scatter(points[:, 0], points[:, 1], s=10, color="red", alpha=0.5, label="Puntos de muestreo")


# Configurar etiquetas de los ejes
ax.set_xlabel("x (posición)")
ax.set_ylabel("t (tiempo)")

# Título del gráfico
ax.set_title("Difusión 1D: Geometría y puntos de muestreo")

# Agregar una cuadrícula para mejor legibilidad
ax.grid(True, ls="--", alpha=0.3)

# Agregar leyenda
ax.legend()

# Mostrar el gráfico
plt.show()

In [ ]:
# ============================================================================
# CONSTRUCCIÓN Y ENTRENAMIENTO DE LA RED NEURONAL
# ============================================================================

# Crear la arquitectura de la red neuronal
# FNN: Feedforward Neural Network
# Estructura: [2 entradas] -> [128 neuronas] -> [128 neuronas] -> [1 salida]
# - Entradas: coordenadas (x, t)
# - Capas ocultas: dos capas con 128 neuronas cada una
# - Función de activación: tanh (tangente hiperbólica)
# - Inicialización de pesos: Glorot normal (Xavier normal)
net = dde.nn.FNN([2] + [128] * 4 + [1], "tanh", "Glorot normal")

# Crear el modelo que combina la red neuronal con el problema de datos
model = dde.Model(data, net)

# Calcular la cantidad de parámetros entrenables en la red neuronal
num_params = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(f"Número de parámetros entrenables en la red neuronal: {num_params}")

# Compilar el modelo con el optimizador Adam
# - optimizer: "adam" (Adam optimizer)
# - lr: tasa de aprendizaje = 0.0001
# - external_trainable_variables: [alpha] es el parámetro externo a entrenar
#   (la difusividad térmica, que no es parte de la red neuronal)
model.compile("adam", lr=1e-4, external_trainable_variables=[alpha], loss_weights=[5, 10, 1, 1])

# Entrenar el modelo durante 10000 épocas (iteraciones completas sobre los datos)
# - epochs: número total de épocas de entrenamiento
# - display_every: mostrar información del entrenamiento cada 100 épocas
train_loss_history = model.train(epochs=50000, display_every=100)

In [ ]:
# ============================================================================
# REFINAMIENTO ADICIONAL CON L-BFGS (BFGS Limitado en Memoria)
# ============================================================================

# Recompilar el modelo con el optimizador L-BFGS para refinamiento
# L-BFGS es un método de optimización más sofisticado que converge a soluciones
# más precisas comparado con Adam, pero es más lento
# - external_trainable_variables=[alpha]: mantener α como variable entrenable

dde.optimizers.LBFGS_options["ftol"] = 1e-15  # Tolerancia para la convergencia de L-BFGS
model.compile("L-BFGS", external_trainable_variables=[alpha])

# Entrenar nuevamente con L-BFGS para refinamiento fino
# Esto realiza un refinamiento fino de la solución obtenida con Adam
model.train()

# Extraer y mostrar el valor estimado de α (difusividad térmica)
# Manejo de diferentes tipos de variables según la implementación de DeepXDE
print("Difusividad térmica estimada (α):", alpha.value if hasattr(alpha, "value") else alpha)

In [ ]:
# ============================================================================
# EVALUACIÓN DE PREDICCIONES Y COMPARACIÓN CON SOLUCIÓN ANALÍTICA
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt

# ============================================================================
# EXTRACCIÓN DEL PARÁMETRO ENTRENADO (α)
# ============================================================================

# Obtener el valor de α entrenado, manejando diferentes tipos de variables
# Se verifica si α tiene un atributo 'value' (caso DeepXDE típico)
alpha_raw = alpha.value if hasattr(alpha, "value") else alpha

# Si α está en GPU o como tensor PyTorch, convertir a valor numérico en CPU
if hasattr(alpha_raw, "detach"):
    alpha_val = float(alpha_raw.detach().cpu().item())
else:
    # Si es un array NumPy o valor escalar
    alpha_val = float(np.asarray(alpha_raw).squeeze())

# ============================================================================
# CREACIÓN DE UNA MALLA REGULAR PARA EVALUACIÓN
# ============================================================================

# Número de puntos en cada dirección (x y t)
n = 100

# Crear vectores de posición equiespaciados
xv = np.linspace(0.0, 1.0, n)  # 100 puntos en [0, 1] para x
tv = np.linspace(0.0, 1.0, n)  # 100 puntos en [0, 1] para t

# Crear una malla bidimensional (x, t) usando meshgrid
X, T = np.meshgrid(xv, tv)

# Reshape para crear array de coordenadas (N, 2) con pares (x, t)
XT = np.hstack([X.reshape(-1, 1), T.reshape(-1, 1)])

# ============================================================================
# EVALUACIÓN DEL MODELO Y CÁLCULO DE LA SOLUCIÓN ANALÍTICA
# ============================================================================

# Obtener predicciones de la red neuronal en los puntos de la malla
# y reshapear al formato de malla 2D (n, n)
u_pred = model.predict(XT).reshape(n, n)

# Calcular la solución analítica exacta usando la fórmula conocida
# u(x,t) = exp(-α π² t) * sin(π x)
u_true = np.exp(-alpha_val * np.pi**2 * T) * np.sin(np.pi * X)

# Calcular el error absoluto punto a punto
error_abs = np.abs(u_pred - u_true)

# ============================================================================
# DETERMINACIÓN DE ESCALA DE COLOR COMPARTIDA
# ============================================================================

# Encontrar los valores mínimo y máximo entre predicción y solución analítica
# para usar la misma escala de color en ambas visualizaciones
vmin = min(u_pred.min(), u_true.min())
vmax = max(u_pred.max(), u_true.max())

# ============================================================================
# VISUALIZACIÓN DE RESULTADOS
# ============================================================================

# Crear figura con 3 subgráficos (1 fila, 3 columnas)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)

# GRÁFICO 1: Predicción del modelo
# Contorno lleno con 60 niveles de coloración
im0 = axes[0].contourf(X, T, u_pred, levels=60, cmap="viridis", vmin=vmin, vmax=vmax)
axes[0].set_title("Predicción del modelo", fontsize=12, fontweight="bold")
axes[0].set_xlabel("x (posición)")
axes[0].set_ylabel("t (tiempo)")

# GRÁFICO 2: Solución analítica
# Contorno lleno con la misma escala que la predicción
im1 = axes[1].contourf(X, T, u_true, levels=60, cmap="viridis", vmin=vmin, vmax=vmax)
axes[1].set_title("Solución analítica", fontsize=12, fontweight="bold")
axes[1].set_xlabel("x (posición)")
axes[1].set_ylabel("t (tiempo)")

# GRÁFICO 3: Error absoluto
# Contorno lleno con escala de colores 'magma' para visualizar mejor el error
im2 = axes[2].contourf(X, T, error_abs, levels=60, cmap="magma")
axes[2].set_title("Error absoluto |u_pred - u_true|", fontsize=12, fontweight="bold")
axes[2].set_xlabel("x (posición)")
axes[2].set_ylabel("t (tiempo)")

# Agregar barras de color
# Barra para predicción y solución analítica (comparten escala)
fig.colorbar(im1, ax=axes[:2], shrink=0.9, label="u(x, t)")

# Barra separada para el error
fig.colorbar(im2, ax=axes[2], shrink=0.9, label="|error|")

# Mostrar la figura
plt.show()